# MLP Baseline: Train and Common Evaluation

Notebook này chạy riêng MLP baseline cho bài toán recommendation và giữ cấu hình chính đồng bộ với DQN:

- State: 5 item gần nhất.
- Output/action space: toàn bộ item id trong train/validation/test.
- Embedding dim: 32, hidden dim: 128, Top-K: 5, learning rate: `1e-4`.
- Train bằng supervised next-item prediction trên `train_history.pkl`.
- Chọn checkpoint tốt nhất chỉ bằng validation, không dùng test để chọn model.
- Final test dùng deterministic sliding windows, valid-action mask, reward và metric giống `evaluation.evaluate_models_common`.

Bật `QUICK_RUN=True` để kiểm tra pipeline nhanh trước khi train đầy đủ.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / 'baseline').is_dir() and (path / 'evaluation').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Không tìm thấy project root.')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

## 1. Cấu hình

`ACTION_DIM` sẽ được suy ra từ cả ba split để khớp global action space. Với dữ liệu hiện tại giá trị mong đợi là `1000`.

In [ ]:
QUICK_RUN = False
DEVICE = 'auto'
SEED = 42

STATE_SIZE = 5
TOP_K = 5
EMBEDDING_DIM = 32
HIDDEN_DIMS = '128'
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
EPOCHS = 30
PATIENCE = 5

TRAIN_PATH = Path('data/processed/train_history.pkl')
VAL_PATH = Path('data/processed/val_history.pkl')
TEST_PATH = Path('data/processed/test_history.pkl')
MLP_MODEL_NAME = 'mlp_common_baseline.pth'
MLP_MODEL_PATH = Path('outputs/checkpoints') / MLP_MODEL_NAME

# Optional direct comparison. Use recent_boost=2 or 5 only with the matching DQN checkpoint.
DQN_MODEL_PATH = Path('outputs/checkpoints/dqn_pure_stable_best.pth')
DQN_RECENT_BOOST = 0.0
RUN_DQN_COMPARISON = not QUICK_RUN

RUN_EPOCHS = 2 if QUICK_RUN else EPOCHS
MAX_TRAIN_SAMPLES = 100_000 if QUICK_RUN else None
MAX_EVAL_SAMPLES = 10_000 if QUICK_RUN else None
MAX_COMMON_WINDOWS = 10_000 if QUICK_RUN else None

## 2. Kiểm tra dữ liệu và global action space

Các split là temporal train/validation/test giống pipeline DQN. MLP chỉ học từ train; validation dùng để chọn checkpoint; test chỉ chạy sau khi model đã được chọn.

In [ ]:
from baseline.baseline_train_v3 import load_history_pickle

split_paths = {'train': TRAIN_PATH, 'validation': VAL_PATH, 'test': TEST_PATH}
summary_rows = []
global_max_item = -1

for split, path in split_paths.items():
    histories = load_history_pickle(path)
    interactions = sum(len(history) for history in histories)
    max_item = max(int(history.max()) for history in histories)
    unique_items = len({int(item) for history in histories for item in history})
    global_max_item = max(global_max_item, max_item)
    summary_rows.append({
        'split': split,
        'histories': len(histories),
        'interactions': interactions,
        'unique_items': unique_items,
        'max_item_id': max_item,
    })
    del histories

ACTION_DIM = global_max_item + 1
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print('GLOBAL ACTION_DIM:', ACTION_DIM)

## 3. Train MLP baseline

MLP nhận embedding của 5 item gần nhất, flatten, đi qua hidden layer 128 và dự đoán item tiếp theo bằng cross-entropy. Script lưu checkpoint tốt nhất theo validation HitRate@5, training log và learning curves.

In [ ]:
train_command = [
    sys.executable, '-m', 'baseline.baseline_train_v3',
    '--train-data-path', str(TRAIN_PATH),
    '--val-data-path', str(VAL_PATH),
    '--test-data-path', str(TEST_PATH),
    '--model-name', MLP_MODEL_NAME,
    '--state-size', str(STATE_SIZE),
    '--top-k', str(TOP_K),
    '--action-dim', str(ACTION_DIM),
    '--epochs', str(RUN_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--embedding-dim', str(EMBEDDING_DIM),
    '--hidden-dims', HIDDEN_DIMS,
    '--lr', str(LEARNING_RATE),
    '--patience', str(PATIENCE),
    '--seed', str(SEED),
    '--device', DEVICE,
]
if MAX_TRAIN_SAMPLES is not None:
    train_command += ['--max-train-samples', str(MAX_TRAIN_SAMPLES)]
if MAX_EVAL_SAMPLES is not None:
    train_command += ['--max-eval-samples', str(MAX_EVAL_SAMPLES)]

print(' '.join(train_command))
subprocess.run(train_command, check=True)

## 4. Xem learning curves và kết quả supervised test

Các metric ở cell này dùng immediate-next-item target. Phần đánh giá chung next-5-window để so sánh công bằng với DQN nằm ở cell kế tiếp.

In [ ]:
history_path = Path('outputs/logs/train_mlp_history_baseline.csv')
supervised_test_path = Path('outputs/logs/mlp_final_test_results.csv')

history_df = pd.read_csv(history_path)
display(history_df.tail())
display(pd.read_csv(supervised_test_path))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train loss')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='validation loss')
axes[0].set_title('MLP loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
metric = f'val_hitrate@{TOP_K}'
axes[1].plot(history_df['epoch'], history_df[metric], label=metric)
axes[1].set_title('Validation HitRate')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Final MLP test trên common deterministic environment

Evaluator này dùng đúng implementation window, valid-action mask, reward và metric từ `evaluation.evaluate_models_common`. Vì vậy kết quả có thể đặt cạnh DQN mà không bị lệch tập test hoặc công thức metric.

In [ ]:
mlp_eval_command = [
    sys.executable, '-m', 'evaluation.evaluate_mlp_common',
    '--data-path', str(TEST_PATH),
    '--model-path', str(MLP_MODEL_PATH),
    '--state-size', str(STATE_SIZE),
    '--top-k', str(TOP_K),
    '--device', DEVICE,
]
if MAX_COMMON_WINDOWS is not None:
    mlp_eval_command += ['--max-windows', str(MAX_COMMON_WINDOWS)]

print(' '.join(mlp_eval_command))
subprocess.run(mlp_eval_command, check=True)
display(pd.read_csv('outputs/logs/mlp_common_test_results.csv'))

## 6. Tùy chọn: so sánh trực tiếp MLP và DQN

Cell này chỉ chạy khi checkpoint DQN tồn tại. Hai model được chấm trên chính xác cùng từng test state và next-5 target.

In [ ]:
if RUN_DQN_COMPARISON and DQN_MODEL_PATH.is_file():
    compare_command = [
        sys.executable, '-m', 'evaluation.evaluate_models_common',
        '--data-path', str(TEST_PATH),
        '--mlp-model-path', str(MLP_MODEL_PATH),
        '--dqn-model-path', str(DQN_MODEL_PATH),
        '--state-size', str(STATE_SIZE),
        '--top-k', str(TOP_K),
        '--recent-boost', str(DQN_RECENT_BOOST),
        '--device', DEVICE,
    ]
    print(' '.join(compare_command))
    subprocess.run(compare_command, check=True)
    comparison_df = pd.read_csv('outputs/logs/common_model_comparison.csv')
    display(comparison_df)
elif not RUN_DQN_COMPARISON:
    print('Bỏ qua so sánh DQN trong QUICK_RUN. Đặt RUN_DQN_COMPARISON=True để chạy.')
else:
    print(f'Bỏ qua so sánh DQN vì chưa có checkpoint: {DQN_MODEL_PATH}')

## File kết quả

- Best MLP checkpoint: `outputs/checkpoints/mlp_common_baseline.pth`
- Training history: `outputs/logs/train_mlp_history_baseline.csv`
- MLP common test: `outputs/logs/mlp_common_test_results.csv` và `.md`
- Optional MLP/DQN comparison: `outputs/logs/common_model_comparison.csv` và `.md`

Các file trong `outputs/` được `.gitignore` để tránh commit checkpoint và kết quả chạy máy cá nhân.